In [9]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.lines import Line2D

sns.set_theme(style="whitegrid", context="talk", font="DejaVu Sans")

out_dir = Path("output")
out_dir.mkdir(exist_ok=True)

results = pd.read_csv("first_crossing_results.csv")

results["pred_prob"] = results["min_bid"].astype(float).clip(0, 1)
results = results[(results["pred_prob"] >= 0) & (results["pred_prob"] <= 0.05)].copy()

results["bin"] = pd.cut(
    results["pred_prob"],
    bins=[0, 0.01, 0.02, 0.03, 0.04, 0.05],
    include_lowest=True,
    labels=["0–1%", "1–2%", "2–3%", "3–4%", "4–5%"]
)

bin_table = results.groupby("bin", observed=True).agg(
    n=("resolved_yes", "size"),
    avg_predicted=("pred_prob", "mean"),
    actual_yes_rate=("resolved_yes", "mean")
).reset_index()

bin_table.to_csv(out_dir / "bin_check_table.csv", index=False)

rng = np.random.default_rng(42)
ci_rows = []
B = 1000

for b in bin_table["bin"].astype(str):
    sub = results.loc[results["bin"].astype(str) == b, "resolved_yes"].astype(int).to_numpy()
    if len(sub) == 0:
        continue
    boot = np.array([rng.choice(sub, size=len(sub), replace=True).mean() for _ in range(B)])
    ci_rows.append({
        "bin": b,
        "ci_low": np.quantile(boot, 0.025),
        "ci_high": np.quantile(boot, 0.975),
        "boot_mean": boot.mean()
    })

ci_df = pd.DataFrame(ci_rows)
ci_df.to_csv(out_dir / "bootstrap_cis.csv", index=False)

plot_df = bin_table.merge(ci_df, on="bin", how="left")

fig, ax = plt.subplots(figsize=(8, 7))

ax.plot([0, 0.05], [0, 0.05], "--", color="gray", linewidth=2)
ax.errorbar(
    plot_df["avg_predicted"],
    plot_df["actual_yes_rate"],
    yerr=[
        plot_df["actual_yes_rate"] - plot_df["ci_low"],
        plot_df["ci_high"] - plot_df["actual_yes_rate"]
    ],
    fmt="o",
    markersize=10,
    capsize=5,
    color="#2C7FB8",
    ecolor="#2C7FB8",
    linewidth=2
)

for _, r in plot_df.iterrows():
    ax.annotate(
        r["bin"],
        (r["avg_predicted"], r["actual_yes_rate"]),
        textcoords="offset points",
        xytext=(8, 6),
        fontsize=10
    )

cal_line = Line2D([0], [0], linestyle="--", color="gray", linewidth=2)
cal_point = Line2D([0], [0], marker="o", linestyle="None", color="#2C7FB8", markersize=8)

ax.set_xlabel("Average predicted probability")
ax.set_ylabel("Observed YES rate")
ax.set_title("Calibration Plot for Markets Crossing Below 5%", pad=12, weight="bold")
ax.set_xlim(0, 0.055)
ax.set_ylim(0, max(0.06, float(np.nanmax(plot_df["actual_yes_rate"].values)) + 0.01))
ax.legend(
    handles=[cal_line, cal_point],
    labels=["Perfect calib.", "Bin estimate"],
    loc="upper left",
    bbox_to_anchor=(0.02, 0.98),
    frameon=False,
    fontsize=12
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(out_dir / "calibration_plot.png", dpi=300, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(plot_df))
width = 0.35

ax.bar(
    x - width / 2,
    plot_df["avg_predicted"],
    width,
    color="#4C78A8",
    edgecolor="black"
)
ax.bar(
    x + width / 2,
    plot_df["actual_yes_rate"],
    width,
    color="#F58518",
    edgecolor="black"
)
ax.errorbar(
    x + width / 2,
    plot_df["actual_yes_rate"],
    yerr=[
        plot_df["actual_yes_rate"] - plot_df["ci_low"],
        plot_df["ci_high"] - plot_df["actual_yes_rate"]
    ],
    fmt="none",
    ecolor="black",
    capsize=4
)

bar_pred = Line2D([0], [0], color="#4C78A8", linewidth=8)
bar_obs = Line2D([0], [0], color="#F58518", linewidth=8)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["bin"].astype(str))
ax.set_ylabel("Probability")
ax.set_title("Observed vs Predicted Probability by Bin", pad=12, weight="bold")
ax.legend(
    handles=[bar_pred, bar_obs],
    labels=["Avg pred.", "Observed YES"],
    loc="upper left",
    bbox_to_anchor=(0.65, 0.98),
    frameon=False,
    fontsize=12
)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(out_dir / "bin_check_barplot.png", dpi=300, bbox_inches="tight")
plt.close(fig)